In [ ]:
import os
import glob
import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid

from compressai.losses import RateDistortionLoss
from pytorch_msssim import ms_ssim
import lpips

from model import AttentionGuidedSwinCompression


print("All libraries loaded successfully!")


# -------------------------
# Hyperparameters
# -------------------------
EPOCHS = 50
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
AUX_LEARNING_RATE = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_SIZE = 512

DATASET_PATH = "/kaggle/input/datasets/jeevajoji/uvg-honeybee-512x512/honeybee_512_crop/"


# -------------------------
# Dataset
# -------------------------
class UVGDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, "*.png")))
        self.transform = transform

        if len(self.image_paths) == 0:
            print(f"Warning: No PNG images found in {root_dir}")
        else:
            print(f"Found {len(self.image_paths)} images.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, self.image_paths[idx]


transform = transforms.Compose([
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.ToTensor()
])

train_dataset = UVGDataset(DATASET_PATH, transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


# -------------------------
# LPIPS
# -------------------------
loss_fn_vgg = lpips.LPIPS(net="vgg").to(DEVICE)
loss_fn_vgg.eval()


# -------------------------
# Optimizer setup
# -------------------------
def configure_optimizers(model, learning_rate=1e-4, aux_learning_rate=1e-3):
    """
    CompressAI models need:
    1. main optimizer for normal model parameters
    2. auxiliary optimizer for entropy bottleneck quantiles
    """

    parameters = {
        name
        for name, param in model.named_parameters()
        if param.requires_grad and not name.endswith(".quantiles")
    }

    aux_parameters = {
        name
        for name, param in model.named_parameters()
        if param.requires_grad and name.endswith(".quantiles")
    }

    params_dict = dict(model.named_parameters())

    optimizer = optim.Adam(
        (params_dict[name] for name in sorted(parameters)),
        lr=learning_rate
    )

    aux_optimizer = optim.Adam(
        (params_dict[name] for name in sorted(aux_parameters)),
        lr=aux_learning_rate
    )

    return optimizer, aux_optimizer


# -------------------------
# Metrics
# -------------------------
@torch.no_grad()
def compute_metrics(x, x_hat, bpp_loss):
    x_hat = x_hat.clamp(0, 1)

    mse = torch.mean((x - x_hat) ** 2).item()
    psnr = 10 * math.log10(1.0 / mse) if mse > 0 else 100.0

    msssim_val = ms_ssim(
        x_hat,
        x,
        data_range=1.0,
        size_average=True
    ).item()

    x_lpips = (x * 2.0) - 1.0
    x_hat_lpips = (x_hat * 2.0) - 1.0

    lpips_val = loss_fn_vgg(
        x_lpips,
        x_hat_lpips
    ).mean().item()

    return bpp_loss.item(), psnr, msssim_val, lpips_val


@torch.no_grad()
def save_visual_comparison(original, reconstructed, epoch, lmbda, out_dir="outputs"):
    os.makedirs(out_dir, exist_ok=True)

    orig_img = original[0].cpu()
    recon_img = reconstructed[0].cpu().clamp(0, 1)

    grid = make_grid([orig_img, recon_img], nrow=2)
    grid_np = grid.permute(1, 2, 0).numpy()

    plt.figure(figsize=(10, 5))
    plt.imshow(grid_np)
    plt.title(f"Epoch {epoch} | Lambda={lmbda} | Left: Original, Right: Reconstructed")
    plt.axis("off")

    out_path = f"{out_dir}/epoch_{epoch}_lambda_{lmbda}.png"
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

    return out_path


@torch.no_grad()
def save_attention_map(spatial_map, epoch, lmbda, out_dir="outputs"):
    os.makedirs(out_dir, exist_ok=True)

    attn = spatial_map[0, 0].detach().cpu().numpy()

    plt.figure(figsize=(5, 5))
    plt.imshow(attn, cmap="viridis")
    plt.colorbar()
    plt.title(f"Decoder-Available Gain Map | Epoch {epoch} | Lambda={lmbda}")
    plt.axis("off")

    out_path = f"{out_dir}/attention_epoch_{epoch}_lambda_{lmbda}.png"
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

    return out_path


# -------------------------
# Training function
# -------------------------
def train_model_for_lambda(lmbda, epochs=EPOCHS):
    print(f"\n--- Training model for Lambda = {lmbda} ---")

    model = AttentionGuidedSwinCompression(N=128, M=192).to(DEVICE)

    optimizer, aux_optimizer = configure_optimizers(
        model,
        learning_rate=LEARNING_RATE,
        aux_learning_rate=AUX_LEARNING_RATE
    )

    criterion = RateDistortionLoss(lmbda=lmbda)

    metrics_log = []

    for epoch in range(1, epochs + 1):
        model.train()

        epoch_loss = 0.0
        epoch_bpp = 0.0
        epoch_psnr = 0.0
        epoch_msssim = 0.0
        epoch_lpips = 0.0
        epoch_aux = 0.0

        for i, (images, _) in enumerate(train_loader):
            images = images.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            aux_optimizer.zero_grad()

            out_net = model(images)
            out_criterion = criterion(out_net, images)

            out_criterion["loss"].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            # Important CompressAI auxiliary loss
            aux_loss = model.aux_loss()
            aux_loss.backward()
            aux_optimizer.step()

            with torch.no_grad():
                bpp, psnr, msssim, lpips_val = compute_metrics(
                    images,
                    out_net["x_hat"],
                    out_criterion["bpp_loss"]
                )

            epoch_loss += out_criterion["loss"].item()
            epoch_bpp += bpp
            epoch_psnr += psnr
            epoch_msssim += msssim
            epoch_lpips += lpips_val
            epoch_aux += aux_loss.item()

        num_batches = len(train_loader)

        avg_loss = epoch_loss / num_batches
        avg_bpp = epoch_bpp / num_batches
        avg_psnr = epoch_psnr / num_batches
        avg_msssim = epoch_msssim / num_batches
        avg_lpips = epoch_lpips / num_batches
        avg_aux = epoch_aux / num_batches

        print(
            f"Epoch {epoch}/{epochs} | "
            f"Loss: {avg_loss:.4f} | "
            f"Aux: {avg_aux:.4f} | "
            f"BPP: {avg_bpp:.4f} | "
            f"PSNR: {avg_psnr:.2f} | "
            f"MS-SSIM: {avg_msssim:.4f} | "
            f"LPIPS: {avg_lpips:.4f}"
        )

        model.eval()

        with torch.no_grad():
            saved_image_path = save_visual_comparison(
                images,
                out_net["x_hat"],
                epoch,
                lmbda
            )

            attention_path = save_attention_map(
                out_net["spatial_map"],
                epoch,
                lmbda
            )

        metrics_log.append({
            "Lambda": lmbda,
            "Epoch": epoch,
            "Total_Loss": avg_loss,
            "Aux_Loss": avg_aux,
            "BPP": avg_bpp,
            "PSNR": avg_psnr,
            "MS_SSIM": avg_msssim,
            "LPIPS": avg_lpips,
            "Image_File": saved_image_path,
            "Attention_Map_File": attention_path
        })

        # Save checkpoint every 10 epochs
        if epoch % 10 == 0:
            checkpoint_path = f"swin_compress_lambda_{lmbda}_epoch_{epoch}.pth"
            torch.save({
                "epoch": epoch,
                "lambda": lmbda,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "aux_optimizer_state_dict": aux_optimizer.state_dict(),
                "metrics": metrics_log
            }, checkpoint_path)

    # Important for entropy coding after training
    model.update()

    final_model_path = f"swin_compress_lambda_{lmbda}_final.pth"
    torch.save(model.state_dict(), final_model_path)

    return metrics_log


# -------------------------
# Run training
# -------------------------
lambdas = [0.0018, 0.0035, 0.0067, 0.0130, 0.0250]

all_metrics = []

for lmbda in lambdas:
    logs = train_model_for_lambda(lmbda, epochs=EPOCHS)
    all_metrics.extend(logs)

df = pd.DataFrame(all_metrics)
df.to_csv("compression_metrics_log.csv", index=False)

print("Training complete. Metrics saved to compression_metrics_log.csv")


# -------------------------
# Markdown visual report
# -------------------------
markdown_content = "# Training Log with Visual Comparisons\n\n"

for _, row in df.iterrows():
    markdown_content += f"## Epoch {int(row['Epoch'])} | Lambda {row['Lambda']}\n\n"
    markdown_content += f"- **BPP:** {row['BPP']:.4f}\n"
    markdown_content += f"- **PSNR:** {row['PSNR']:.2f}\n"
    markdown_content += f"- **MS-SSIM:** {row['MS_SSIM']:.4f}\n"
    markdown_content += f"- **LPIPS:** {row['LPIPS']:.4f}\n\n"
    markdown_content += f"### Reconstruction\n\n"
    markdown_content += f"![Comparison]({row['Image_File']})\n\n"
    markdown_content += f"### Decoder-Available Gain Map\n\n"
    markdown_content += f"![Attention Map]({row['Attention_Map_File']})\n\n"

with open("training_report.md", "w") as f:
    f.write(markdown_content)

print("Markdown visual report saved to training_report.md")